# 02_model_keras_rs.ipynb
**Sistema de Recomendación con Two-Tower Model**

Este notebook entrena el modelo de **Collaborative Filtering** que genera los embeddings usados en notebooks posteriores.

## Objetivo:
1. Entrenar modelo Two-Tower con interacciones usuario-item
2. Generar embeddings de 64 dimensiones para usuarios y juegos
3. Guardar `item_embeddings_rs.npy` y `user_embeddings_rs.npy`

## Output:
- `../Data/item_embeddings_rs.npy` - Embeddings de juegos (usado en notebooks 03-09)
- `../Data/user_embeddings_rs.npy` - Embeddings de usuarios

In [1]:
import os
os.environ["KERAS_BACKEND"] = "jax"
import pandas as pd
import numpy as np
import json
import keras
#keras.config.backend = "jax"
import keras_rs

# Semilla para reproducibilidad del Two-Tower
import os as _os
_os.environ["PYTHONHASHSEED"] = "42"
np.random.seed(42)
keras.utils.set_random_seed(42)

INTERACTIONS = "../Data/interactions.parquet"
USERMAP = "../Data/user2idx.json"
ITEMMAP = "../Data/item2idx.json"

df = pd.read_parquet(INTERACTIONS)

with open(USERMAP, "r") as f:
    user2idx = {k:int(v) for k,v in json.load(f).items()}
with open(ITEMMAP, "r") as f:
    item2idx = {k:int(v) for k,v in json.load(f).items()}

num_users = len(user2idx)
num_items = len(item2idx)

print(f"Users: {num_users}, Items: {num_items}")
print(df.shape)
df.head()

C:\Users\matia\anaconda3\envs\VG_RS\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


Users: 25458, Items: 3682
(59305, 4)


,user_id,item_id,user_idx,item_idx
0,76561197970982479,1250,0,0
1,76561197970982479,22200,0,1
2,76561197970982479,43110,0,2
3,js41637,251610,1,3
4,js41637,227300,1,4


## Cargar Datos de Interacciones

In [2]:
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
df_valid, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

def build_dataset(df):
    return {
        "user_id": df["user_idx"].values.astype("int32"),
        "item_id": df["item_idx"].values.astype("int32"),
        "label": np.ones(len(df), dtype="float32")
    }

train_dict = build_dataset(df_train)
valid_dict = build_dataset(df_valid)
test_dict  = build_dataset(df_test)

## Split Train/Valid/Test

In [3]:
# Verificar qué tiene keras_rs disponible
import keras_rs
print("Componentes de keras_rs.layers:")
print(dir(keras_rs.layers))
print("\nComponentes de keras_rs.models:")
print(dir(keras_rs.models) if hasattr(keras_rs, 'models') else "No models module")

Componentes de keras_rs.layers:
['BruteForceRetrieval', 'DistributedEmbedding', 'DotInteraction', 'EmbedReduce', 'FeatureConfig', 'FeatureCross', 'HardNegativeMining', 'RemoveAccidentalHits', 'Retrieval', 'SamplingProbabilityCorrection', 'TableConfig', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']

Componentes de keras_rs.models:
No models module


In [4]:
# Modelo usando keras_rs
from keras import layers, Model
import keras_rs

# Inputs
user_input = layers.Input(shape=(), name='user_id', dtype='int32')
item_input = layers.Input(shape=(), name='item_id', dtype='int32')

# Embeddings usando capas estándar (keras_rs no tiene capa simple de embedding)
user_embedding_layer = layers.Embedding(
    input_dim=num_users,
    output_dim=64,
    name='user_embedding'
)

item_embedding_layer = layers.Embedding(
    input_dim=num_items,
    output_dim=64,
    name='item_embedding'
)

user_emb = user_embedding_layer(user_input)
item_emb = item_embedding_layer(item_input)

# Usar DotInteraction de keras_rs para calcular la similitud
dot_interaction = keras_rs.layers.DotInteraction(name='dot_interaction')
interaction_output = dot_interaction([user_emb, item_emb])

# Sigmoid para convertir a probabilidad
output = layers.Activation('sigmoid')(interaction_output)

# Crear modelo
model = Model(inputs=[user_input, item_input], outputs=output)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=['accuracy']
)

print(model.summary())

# Entrenar
history = model.fit(
    [train_dict["user_id"], train_dict["item_id"]],
    train_dict["label"],
    validation_data=(
        [valid_dict["user_id"], valid_dict["item_id"]],
        valid_dict["label"]
    ),
    epochs=10,
    batch_size=1024,
    verbose=1
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_id             │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_id             │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 64)        │  1,629,312 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 64)        │    235,648 │ item_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_interaction     │ (None, 1)         │          0 │ user_embedding[0… │
│ (DotInteraction)    │                   │            │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 1)         │          0 │ dot_interaction[… │
│ (Activation)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,864,960 (7.11 MB)

 Trainable params: 1,864,960 (7.11 MB)

 Non-trainable params: 0 (0.00 B)

...

11/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9999 - loss: 0.3334

21/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9999 - loss: 0.3300

31/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.3275

41/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.3255

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.3162 - val_accuracy: 0.7297 - val_loss: 0.6112


## Construir y Entrenar Modelo Two-Tower

**Arquitectura:**
- User Embedding: 64 dims
- Item Embedding: 64 dims
- Dot Product Interaction (keras_rs)
- Sigmoid output (probabilidad de interacción)

**Training:**
- Loss: Binary Crossentropy
- Optimizer: Adam
- Epochs: 10
- Batch size: 1024

In [5]:
from sklearn.metrics import ndcg_score

def ndcg_user(model, user_id, positives, num_items, k=10, neg=99):
    negatives = np.random.choice(
        list(set(range(num_items)) - set(positives)),
        size=neg, replace=False
    )
    candidates = np.array(list(positives) + list(negatives))

    preds = model.predict({
        "user_id": np.array([user_id]*len(candidates)),
        "item_id": candidates
    }, verbose=0).flatten()

    y_true = np.array([1] + [0]*neg)
    y_pred = preds

    return ndcg_score([y_true], [y_pred], k=k)

users_eval = df_test["user_idx"].unique()[:500]
scores = []

for u in users_eval:
    positives = df_test[df_test["user_idx"] == u]["item_idx"].tolist()
    if len(positives) == 0:
        continue
    scores.append(ndcg_user(model, u, positives[:1], num_items))

print("NDCG@10 =", np.mean(scores))

NDCG@10 = 0.3143504408713464


## Evaluación: NDCG@10

Evalúa el modelo calculando NDCG@10 en 500 usuarios del test set.

In [6]:
# Guardar embeddings
user_emb = model.get_layer('user_embedding').get_weights()[0]
item_emb = model.get_layer('item_embedding').get_weights()[0]

np.save("../Data/user_embeddings_rs.npy", user_emb)
np.save("../Data/item_embeddings_rs.npy", item_emb)

print(f"User embeddings shape: {user_emb.shape}")
print(f"Item embeddings shape: {item_emb.shape}")
print("Embeddings saved!")

User embeddings shape: (25458, 64)
Item embeddings shape: (3682, 64)
Embeddings saved!


In [7]:
import os

print("="*80)
print("VERIFICACIÓN DE EMBEDDINGS GENERADOS")
print("="*80)

# Verificar que se guardaron correctamente
files_to_check = [
    ("../Data/user_embeddings_rs.npy", "User embeddings", user_emb.shape),
    ("../Data/item_embeddings_rs.npy", "Item embeddings", item_emb.shape),
]

for filepath, description, expected_shape in files_to_check:
    exists = os.path.exists(filepath)
    status = "✅" if exists else "❌"
    size = f"{os.path.getsize(filepath) / 1024:.1f} KB" if exists else "N/A"
    print(f"{status} {filepath:40s}")
    print(f"   {description:20s} - Shape: {expected_shape} - Size: {size}")

print("\n" + "="*80)
print("RESUMEN DEL MODELO")
print("="*80)
print(f"Arquitectura: Two-Tower Model (Collaborative Filtering)")
print(f"Embedding dimension: 64")
print(f"Users: {num_users:,}")
print(f"Items: {num_items:,}")
print(f"Training samples: {len(train_dict['user_id']):,}")
print(f"Validation samples: {len(valid_dict['user_id']):,}")
print(f"Test samples: {len(test_dict['user_id']):,}")
print("="*80)

print("\n✅ Embeddings listos para usar en notebooks 03-09")
print("   - item_embeddings_rs.npy → usado como features para regresión")
print("   - user_embeddings_rs.npy → guardado para uso futuro")

VERIFICACIÓN DE EMBEDDINGS GENERADOS
✅ ../Data/user_embeddings_rs.npy          
   User embeddings      - Shape: (25458, 64) - Size: 6364.6 KB
✅ ../Data/item_embeddings_rs.npy          
   Item embeddings      - Shape: (3682, 64) - Size: 920.6 KB

RESUMEN DEL MODELO
Arquitectura: Two-Tower Model (Collaborative Filtering)
Embedding dimension: 64
Users: 25,458
Items: 3,682
Training samples: 47,444
Validation samples: 5,930
Test samples: 5,931

✅ Embeddings listos para usar en notebooks 03-09
   - item_embeddings_rs.npy → usado como features para regresión
   - user_embeddings_rs.npy → guardado para uso futuro


## Verificación Final

## Guardar Embeddings

**IMPORTANTE:** Estos embeddings se usan en todos los notebooks de regresión (03-09)